# Ignite-3B S04 - Cond C gen 3-5 (resume)

Continue Cond C from S03. `--resume` reads state.json + last (gen, cand).

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
HF_REPO = 'iterate-labs-ai/ignite-3b-cond-c'
PREV_KAGGLE_DATASET = 'vitorscrt/ignite-3b-cond-c-s03'
OUT_ROOT = '/kaggle/working/cond_C_run'
GENS_TOTAL = 6   # so this session covers gens 3, 4, 5
CANDS = 8
STEPS = 150

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'unsloth>=2025.1.0' 'trl>=0.12.0' 'vllm>=0.6.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'scipy' 'huggingface_hub' kaggle

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
import subprocess
subprocess.run(['kaggle', 'datasets', 'download', '-d', PREV_KAGGLE_DATASET, '-p', OUT_ROOT, '--unzip', '--force'], check=True)
subprocess.run(['ls', '-la', OUT_ROOT])

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'train.ignite.C_rsi_outer',
       '--base', BASE_MODEL,
       '--dataset-train', 'data/ignite/omni_math_train.jsonl',
       '--dataset-dev', 'data/ignite/omni_math_dev.jsonl',
       '--dataset-val', 'data/ignite/omni_math_val.jsonl',
       '--bench', 'math',
       '--bench-name', 'omni_math',
       '--gens', str(GENS_TOTAL),
       '--cands', str(CANDS),
       '--steps', str(STEPS),
       '--out', OUT_ROOT,
       '--hf-repo', HF_REPO,
       '--resume']
subprocess.run(cmd, check=True)

In [ ]:
import json, subprocess
from pathlib import Path
pub = Path(OUT_ROOT)
(pub / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Ignite-3B Cond C S04', 'id': 'vitorscrt/ignite-3b-cond-c-s04',
    'licenses': [{'name': 'Apache-2.0'}],
}, indent=2))
r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(pub), '-m', 'gen3-5'], capture_output=True, text=True, check=False)
if r.returncode != 0:
    subprocess.run(['kaggle', 'datasets', 'create', '-p', str(pub), '--public'], check=True)